# I help by uploding the pages to Firestore and TOC (can be removed for)


In [ ]:
# @title Imports

from datetime import datetime
import firebase_admin
from firebase_admin import credentials, firestore
from google.cloud.firestore_v1 import FieldFilter
from uuid import uuid4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Preparing TOC

In [ ]:
# @title Inputs (Update Before Run)
BOOK_NAME= "Manuscript"
BOOK_NAME_MODIFIED = BOOK_NAME.replace(" ", "_")

BOOK_PART_NUMBER = 8 # 0 MEANS IT DOESNT BELONG TO A SERIES

BOOK_ID = f"{BOOK_NAME_MODIFIED}_{str(uuid4())[:23]}"
# BOOK_ID = "fiqh_001-123456" #remove

BOOK_TITLE_PRODUCTION= f""
BOOK_SUBTITLE_PRODUCTION= ""
BOOK_AUTHOR="Manuescript"
SEARCH_RANGE=100
TOC_OFFEST=0

BOOK_URL_FIRESTORE =""

print(f"Done Inputs for {BOOK_ID} - part {BOOK_PART_NUMBER} -- {BOOK_NAME}")

Done Inputs for Manuscript_e7a5af48-5530-4bdc-9501 - part 8 -- Manuscript


In [ ]:
# @title Input Files (Update Before Run)
OUTPUT_DRIVE=f"/content/drive/MyDrive/_AgenticIslam/_Parsed Books Production/Manuscript/{BOOK_NAME}"
OUTPUT_DRIVE= "/content/drive/MyDrive/_AgenticIslam/_Parsed Books Production/Manuscript" #remove

INPUT_TOC_PATH = f"/content/Fiqh08-toc.md"

BATCH_RESULTS_PATH = f"/content/Manuscript_1-job_batch_result.json"


In [ ]:
# @title Converting toc.md to toc flat.json (OLD Stars approch)
# # Parse the uploaded TOC file and export a flattened JSON with page ranges and paths.
# import re, json, os
# from dataclasses import dataclass, asdict
# from typing import List, Optional
# import pandas as pd


# # Read file
# with open(INPUT_TOC_PATH, "r", encoding="utf-8") as f:
#     raw = f.read()

# # Patterns to parse lines like:
# # * **Title**: 123
# # with variable indentation (4 spaces per level)
# pat_bold = re.compile(r'^(?P<indent>\s*)\*\s+\*\*(?P<title>.+?)\*\*\s*:\s*(?P<page>\d+)\s*$')
# pat_plain = re.compile(r'^(?P<indent>\s*)\*\s+(?P<title>.+?)\s*:\s*(?P<page>\d+)\s*$')

# dataclass
# class Node:
#     id: str
#     level: int
#     title: str
#     startPage: int
#     endPage: Optional[int] = None
#     parentId: Optional[str] = None
#     pathIds: Optional[List[str]] = None
#     pathTitles: Optional[List[str]] = None

# lines = [ln.rstrip() for ln in raw.splitlines() if ln.strip()]
# nodes: List[Node] = []
# stack: List[Node] = []  # nodes representing current path
# counter = 0

# def level_from_indent(indent: str) -> int:
#     # Assume every 4 spaces increases level by 1, base level = 1
#     spaces = len(indent.replace("\t", "    "))  # normalize any tabs
#     return spaces // 4 + 1

# for ln in lines:
#     m = pat_bold.match(ln) or pat_plain.match(ln)
#     if not m:
#         # Not a TOC line with page number; skip
#         continue

#     indent = m.group("indent")
#     title = m.group("title").strip()
#     page = int(m.group("page"))
#     lvl = level_from_indent(indent)

#     # Adjust stack to current level
#     while stack and stack[-1].level >= lvl:
#         stack.pop()

#     counter += 1
#     node_id = f"item_{counter}"
#     parent_id = stack[-1].id if stack else None

#     # Prepare path (include ancestors + self)
#     current_path_ids = [n.id for n in stack] + [node_id]
#     current_path_titles = [n.title for n in stack] + [title]

#     node = Node(
#         id=node_id,
#         level=lvl,
#         title=title,
#         startPage=page,
#         parentId=parent_id,
#         pathIds=current_path_ids,
#         pathTitles=current_path_titles
#     )
#     nodes.append(node)
#     stack.append(node)

# # Compute endPage for each node
# for i, node in enumerate(nodes):
#     # default end page is BOOK_END
#     end_page = BOOK_END
#     # look for the next item that closes this section (same or higher-level)
#     for j in range(i+1, len(nodes)):
#         nxt = nodes[j]
#         if nxt.level <= node.level:
#             # the current section ends right before the next section starts
#             if nxt.startPage > node.startPage:
#                 end_page = nxt.startPage - 1
#             else:
#                 # same page or earlier (defensive): clamp to same page
#                 end_page = node.startPage
#             break
#     # clamp to at least startPage
#     if end_page < node.startPage:
#         end_page = node.startPage
#     node.endPage = end_page


# # Export to JSON in Drive to save it
# flat = [asdict(n) for n in nodes]
# with open(f"{OUTPUT_DRIVE}/{BOOK_NAME_MODIFIED}_toc_flat.json", "w", encoding="utf-8") as f:
#     json.dump(flat, f, ensure_ascii=False, indent=2)

# # Export to JSON in output for easy access
# with open(f"{BOOK_NAME_MODIFIED}_toc_flat.json", "w", encoding="utf-8") as f:
#     json.dump(flat, f, ensure_ascii=False, indent=2)

# # Show a small preview to the user
# df_preview = pd.DataFrame(flat[:25])

# df_preview

'/Hashit_Ib_3abdeen_toc_flat.json'

In [ ]:
# @title Converting toc.md to toc flat.json (from VS code)
import re
import json

def md_toc_to_flat_json(md_text: str, *, last_item_end_page: int | None = None) -> list[dict]:
    """
    Convert a nested Markdown TOC with bullets into a flat JSON structure.

    Expected line shape (Unicode-safe):
        [spaces] - TITLE : PAGE
    where PAGE is at the *end* of the line and is digits (Arabic or Arabic-Indic).

    Args:
        md_text: The Markdown content as a single string.
        last_item_end_page: Optional explicit end page for the very last item.
            - If provided and >= its start page, the last item's endPage will use this.
            - Otherwise, the last item's endPage defaults to its startPage.

    Returns:
        A list of dicts with keys:
            id, level, title, startPage, endPage, parentId,
            pathIds, pathTitles, is_confident
    """
    # Support Arabic-Indic digits too
    arabic_indic = "٠١٢٣٤٥٦٧٨٩"
    western = "0123456789"
    DIGIT_MAP = str.maketrans(arabic_indic, western)

    # 1) Parse all bullet lines -> collect raw nodes with indent length
    item_re = re.compile(
        r'^(?P<indent>\s*)-\s*(?P<title>.+?)\s*:\s*(?P<page>[0-9\u0660-\u0669]+)\s*$'
    )
    raw = []
    for ln in md_text.splitlines():
        m = item_re.match(ln.rstrip())
        if not m:
            continue
        indent_len = len(m.group('indent').expandtabs(2))  # normalize tabs → spaces
        title = m.group('title').strip()

        # Strip bold markers if the whole title is in **...**
        bold_wrapped = re.fullmatch(r"\*\*(.+?)\*\*", title)
        if bold_wrapped:
            title = bold_wrapped.group(1).strip()

        # Normalize internal whitespace
        title = re.sub(r"\s+", " ", title)

        start_page = int(m.group('page').translate(DIGIT_MAP))
        raw.append({"indent": indent_len, "title": title, "startPage": start_page})

    if not raw:
        return []

    # 2) Infer indentation unit (e.g., 2 spaces) and compute levels
    nonzero_indents = sorted({r["indent"] for r in raw if r["indent"] > 0})
    indent_unit = nonzero_indents[0] if nonzero_indents else 2  # fallback
    for r in raw:
        r["level"] = (r["indent"] // indent_unit) + 1

    # 3) Assign ids
    for i, r in enumerate(raw, start=1):
        r["id"] = f"item_{i}"

    # 4) Compute parentId and path arrays using a level stack
    out = []
    stack = []  # holds indices of ancestors in 'out'
    for r in raw:
        lvl = r["level"]
        while stack and out[stack[-1]]["level"] >= lvl:
            stack.pop()

        parent_id = out[stack[-1]]["id"] if stack else None

        node = {
            "id": r["id"],
            "level": r["level"],
            "title": r["title"],
            "startPage": r["startPage"],
            # endPage assigned in step 5
            "endPage": None,
            "parentId": parent_id,
            # temporary; fill after pushing to stack
            "pathIds": None,
            "pathTitles": None,
            "is_confident": True,
        }

        # Build path from the stack (ancestors) + self
        path_ids = [out[idx]["id"] for idx in stack] + [node["id"]]
        path_titles = [out[idx]["title"] for idx in stack] + [node["title"]]
        node["pathIds"] = path_ids
        node["pathTitles"] = path_titles

        out.append(node)
        stack.append(len(out) - 1)

    # 5) Compute endPage:
    #    endPage = start of next item with level <= current level, minus 1.
    #    If none exists (last item of the whole doc), use provided last_item_end_page
    #    or fall back to its own startPage.
    for i, node in enumerate(out):
        end_page = None
        for j in range(i + 1, len(out)):
            if out[j]["level"] <= node["level"]:
                end_page = out[j]["startPage"] - 1
                break
        if end_page is None:
            if isinstance(last_item_end_page, int) and last_item_end_page >= node["startPage"]:
                end_page = last_item_end_page
            else:
                end_page = node["startPage"]
        node["endPage"] = end_page

    return out


In [ ]:
# ---- parsing to usage ----
from pathlib import Path
md = Path(INPUT_TOC_PATH).read_text(encoding="utf-8")
flat = md_toc_to_flat_json(md)

with open(f"{OUTPUT_DRIVE}/{BOOK_NAME_MODIFIED}_toc_flat.json", "w", encoding="utf-8") as f:
    json.dump(flat, f, ensure_ascii=False, indent=2)

# Export to JSON in output for easy access
with open(f"{BOOK_NAME_MODIFIED}_toc_flat.json", "w", encoding="utf-8") as f:
    json.dump(flat, f, ensure_ascii=False, indent=2)

## Addting TOC to Firestore

In [ ]:
# @title Init Firestore
from typing import Iterable, Dict, Any, Optional, List
import json

import firebase_admin
from firebase_admin import credentials, firestore
# Upload pages
import time
import random
from typing import List, Dict, Iterable, Optional
from google.api_core.exceptions import Aborted, DeadlineExceeded, ServiceUnavailable
from google.cloud.firestore_v1 import FieldFilter

try:
    cred = credentials.Certificate(f"/content/drive/MyDrive/_AgenticIslam/firebase/keys/agentix-islam-service_account.json")
    agenticIslam = firebase_admin.initialize_app(cred, name="AgenticIslam")
    db = firestore.client(app=agenticIslam)
except ValueError:
    # This handles the case where the app is already initialized, for example in a development environment.
    print("Firebase app already initialized.")
db

Firebase app already initialized.


In [ ]:
# @title Adding toc_flat.json to Firestore

def _chunked(iterable: Iterable[Dict[str, Any]], size: int) -> Iterable[List[Dict[str, Any]]]:
    """Yield lists of at most `size` items from `iterable`."""
    batch: List[Dict[str, Any]] = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch


def upload_toc(
        book_id: str,
        book_number: str,
        toc_items: Iterable[Dict[str, Any]],
        *,
        merge: bool = True,
        batch_size: int = 500
) -> Dict[str, Any]:
    """
    Upload a TOC list to Firestore under books/{book_id}/toc.

    - book_id: your book document ID.
    - toc_items: iterable of dicts like:
        {
          "id": "item_1",
          "level": 1,
          "title": "مقدمة الطبعة الجديدة",
          "startPage": 17,
          "endPage": 20,
          "parentId": None,
          "pathIds": ["item_1"],
          "pathTitles": ["مقدمة الطبعة الجديدة"],
        }
    - merge: if True, upserts documents; if False, overwrites completely.

    Returns summary: {"book_id": ..., "written": N, "batches": B}
    """
    col_ref = db.collection("books").document(book_id).collection("parts").document(book_number).collection("toc")

    total_written = 0
    batches = 0

    for group in _chunked(toc_items, batch_size):
        batch = db.batch()
        for entry in group:
            doc_id = entry.get("id")
            if not doc_id:
                # If missing, generate an auto ID but keep the entry as-is otherwise.
                doc_ref = col_ref.document()
            else:
                doc_ref = col_ref.document(str(doc_id))

            # Write the entry as-is. merge=True prevents duplicates on reruns.
            batch.set(doc_ref, entry, merge=merge)
        batch.commit()
        total_written += len(group)
        batches += 1

    return {"book_id": book_id, "written": total_written, "batches": batches}


# ---------- Convenience: load from a JSON file ----------

def upload_toc_from_file(
        book_id: str,
        book_number: str,
        json_path: str,
        *,
        merge: bool = True,
) -> Dict[str, Any]:
    """
    Read a JSON array from `json_path` and upload to books/{book_id}/toc.
    """
    print(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        toc_list = json.load(f)
        if not isinstance(toc_list, list):
            raise ValueError("Expected a JSON array at the root.")
    return upload_toc(
        book_id,
        book_number,
        toc_list,
        merge=merge,
    )

In [ ]:
upload_toc_from_file(BOOK_ID, f"{BOOK_PART_NUMBER}",f"/content/{BOOK_NAME_MODIFIED}_toc_flat.json")

/content/Fiqh_Zhiali_toc_flat.json


{'book_id': 'fiqh_001-123456', 'written': 292, 'batches': 1}

Upload Pages to firestore

In [ ]:
#  @title Read the batched json
import json

file_path = BATCH_RESULTS_PATH

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Successfully loaded data from {file_path}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully loaded data from /content/Manuscript_1-job_batch_result.json


In [ ]:
# @title Process results batch (Parse it)
import json
import re

def parse_and_process_data(json_data):
    """
    Parses a list of JSON objects, extracts specific data, and sums up usage metadata.

    Args:
        json_data: A list of JSON objects containing API response data.

    Returns:
        A dictionary containing a list of extracted pages and a dictionary
        with the summed usage metadata.
    """
    pages_list = []
    total_usage_metadata = {
        "totalTokenCount": 0,
        "candidatesTokenCount": 0,
        "promptTokenCount": 0,
    }

    for item in json_data:
        try:
            # Extract the raw text from the specified path
            raw_text = item.get('response', {}).get('candidates', [{}])[0].get('content', {}).get('parts', [{}])[0].get('text', '{}')

            # Load the text as a JSON object and append it to pages_list
            page_json = json.loads(raw_text)
            pages_list.append(page_json)

            # Sum the usage metadata
            usage_metadata = item.get('response', {}).get('usageMetadata', {})
            total_usage_metadata['totalTokenCount'] += usage_metadata.get('totalTokenCount', 0)
            total_usage_metadata['candidatesTokenCount'] += usage_metadata.get('candidatesTokenCount', 0)
            total_usage_metadata['promptTokenCount'] += usage_metadata.get('promptTokenCount', 0)

        except (json.JSONDecodeError, IndexError, TypeError) as e:
            # Handle potential errors gracefully
            print(f"Error processing item: {item}")
            continue

    return {
        "pages_list": pages_list,
        "Total_usageMetadata": total_usage_metadata
    }

In [ ]:
# Call the function and print the result
output = parse_and_process_data(data)
pages = output["pages_list"]
print(pages[10])
print(output["Total_usageMetadata"])

{}
{'totalTokenCount': 0, 'candidatesTokenCount': 0, 'promptTokenCount': 0}


In [ ]:
# @title Save pages to Firestore
save_json=f"{OUTPUT_DRIVE}/{BOOK_NAME_MODIFIED}_pages.json"

# Export to JSON in output for easy access
with open(save_json, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
# Export to JSON in output for easy access
with open(f"{BOOK_NAME}_pages.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

In [ ]:
# @title Upload pages to firestore
def upload_pages_to_firestore(
        book_id: str,
        book_number: int,
        pages: List[Dict],
        batch_size: int = 450,
        overwrite: bool = False,
        collection_root: str = "books",
        max_retries: int = 5,
        jitter_seconds: float = 0.2,
) -> int:
    """
    يرفع صفحات كتاب (قائمة JSON) إلى Firestore تحت:
        /books/{book_id}/pages/{page_number}

    Args:
        book_id: معرف الكتاب.
        pages: قائمة من القواميس، كل عنصر يحتوي على الأقل:
               {"page_number": int, "page_content": str, "is_missing": bool}
        batch_size: حجم الدفعة (<= 500).
        overwrite: إذا True يستخدم set(.., merge=False) فيستبدل الوثيقة بالكامل.
                   إذا False (الموصى به) يستخدم merge=True للتحديث دون استبدال كامل.
        collection_root: اسم مجموعة الجذر ("books" افتراضيًا).
        max_retries: عدد محاولات إعادة الإرسال عند الأخطاء القابلة لإعادة المحاولة.
        jitter_seconds: اهتزاز زمني صغير لتقليل تضارب الطلبات.

    Returns:
        عدد الوثائق المكتوبة بنجاح.
    """
    if batch_size < 1 or batch_size > 500:
        raise ValueError("batch_size must be between 1 and 500")

    col_ref = db.collection("books").document(book_id).collection("parts").document(f"{book_number}").collection("pages")


    written = 0

    # تحقق بسيط من المدخلات
    cleaned_pages = []
    for p in pages:
        if "page_number" not in p:
            print("^^"*50)
            print(p)
            # raise ValueError("Each page dict must include 'page_number'.")
        cleaned_pages.append(p)

    for chunk in _chunked(cleaned_pages, batch_size):
        attempt = 0
        while True:
            try:
                batch = db.batch()
                for page in chunk:
                    page_num = page.get("page_number", -1)
                    if page_num is None:
                        print("^^"*50)
                        print(page)
                        # raise ValueError("page_number cannot be None")

                    # استخدم page_number كمعرّف الوثيقة لسهولة القراءة/التحديث لاحقًا
                    doc_ref = col_ref.document(str(page_num))

                    notes = page.get("notes", {})  # If 'notes' doesn't exist, return an empty dictionary
                    footnote = notes.get("footnote", "")  # If 'footnote' doesn't exist, return an empty string
                    marginalia = notes.get("marginalia", "") # If 'marginalia' doesn't exist, return an empty string

                    # Your data structure would then be built like this:
                    data = {
                        "page_number": int(page.get("page_number", "-1")), # Use .get() here too for robustness
                        "page_content": page.get("page_content", ""),
                        "footnote": footnote,
                        "marginalia": marginalia,
                    }

                    # يمكنك إضافة حقول إضافية هنا مثل created_at/updated_at باستخدام cloud functions
                    if overwrite:
                        batch.set(doc_ref, data, merge=False)
                    else:
                        batch.set(doc_ref, data, merge=True)

                # تنفيذ الكتابة
                batch.commit()
                written += len(chunk)
                # فاصل صغير اختياري لتجنّب ضغط مرتفع
                time.sleep(random.uniform(0, jitter_seconds))
                break  # خرجنا من حلقة retry لهذه الدفعة

            except (Aborted, DeadlineExceeded, ServiceUnavailable) as e:
                attempt += 1
                if attempt > max_retries:
                    raise RuntimeError(
                        f"Failed to commit a batch after {max_retries} retries"
                    ) from e
                # backoff أُسّي مع اهتزاز
                sleep_s = (2 ** (attempt - 1)) + random.uniform(0, jitter_seconds)
                time.sleep(sleep_s)

    return written


def upload_page_content_to_db(pages: List[Dict], fiqh_book_id: str, book_number: int):
    # حمّل ملف الصفحات الكبير

    count = upload_pages_to_firestore(
        book_id=fiqh_book_id,
        book_number=book_number,
        pages=pages,
        batch_size=450,  # < 500 للتوافق مع حدود Firestore
        overwrite=False  # استخدم merge لتحديث آمن دون استبدال كامل
    )

    print(f"تم رفع {count} صفحة بنجاح.")

In [ ]:
upload_page_content_to_db(pages,BOOK_ID, BOOK_PART_NUMBER)

تم رفع 791 صفحة بنجاح.


In [ ]:
# @title Prepare Book Edition
import json
import os

def create_markdown_book_from_json(output_drive, pages_json_file, book_name_modified):
    """
    Reads a JSON file containing book pages, then formats and writes
    the content to a single Markdown (.md) file.

    The Markdown structure for each page is:
    [page_content]
    [footnotes (if present)]
    [page_number]
    ---
    ---

    Args:
        output_drive (str): The directory path where the JSON file is located
                            and where the output MD file will be saved (e.g., "C:/Output").
        book_name_modified (str): The base name for the book file (e.g., "Mawsu'at_al-Fiqh").
    """
    # 1. Define file paths
    save_json = os.path.join(output_drive,  pages_json_file)
    output_md_file = os.path.join(output_drive, f"{book_name_modified}_book.md")

    # 2. Read the JSON file
    try:
        with open(save_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: JSON file not found at {save_json}")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {save_json}")
        return
    except Exception as e:
        print(f"An unexpected error occurred while reading the JSON: {e}")
        return

    # Check if 'pages_list' key exists in the JSON
    if 'pages_list' not in data or not isinstance(data['pages_list'], list):
        print("Error: JSON does not contain a valid 'pages_list'.")
        return

    # 3. Process and compile the book content
    md_content = []

    # Define the divider as requested (two dividers)
    page_divider = "---\n---\n"

    for page in data['pages_list']:
        # Extract fields, defaulting to empty string for safety
        page_content = page.get('page_content', '').strip()
        page_number = page.get('page_number', '').strip()

        # Extract footnote content
        # Check if 'notes' key exists and is a dictionary
        notes = page.get('notes')
        footnote = ''
        if isinstance(notes, dict):
            # The request only mentioned 'footnotes', so we check 'footnote' within 'notes'
            # and only include it if it's a non-empty string.
            note_content = notes.get('footnote')
            if note_content and isinstance(note_content, str):
                # Add a clear heading for the footnote for better visibility
                footnote = f"\n\n**Footnote:**\n{note_content.strip()}"

            # Note: marginalia is ignored as per your compilation request.

        # Compile the content for the current page in the requested order:
        # 1. page content
        # 2. footnotes
        # 3. page number
        # 4. two dividers

        page_block = f"{page_content}{footnote}\n\n**Page:** {page_number}\n{page_divider}"

        # Add the page block to the list
        md_content.append(page_block)

    # 4. Write the compiled content to the Markdown file
    try:
        # Join all page blocks together. The divider is already at the end of each block,
        # so simply joining them creates the complete book.
        final_md_text = "\n".join(md_content)

        with open(output_md_file, 'w', encoding='utf-8') as f:
            f.write(final_md_text)

        # oone here to see it
        with open(f"{book_name_modified}_book.md", 'w', encoding='utf-8') as f:
            f.write(final_md_text)

        print(f"Successfully created Markdown book file at: {output_md_file}")

    except Exception as e:
        print(f"An error occurred while writing the Markdown file: {e}")

In [ ]:
create_markdown_book_from_json(OUTPUT_DRIVE,f"{BOOK_NAME_MODIFIED}_pages.json",BOOK_NAME_MODIFIED)

Error: JSON file not found at /content/drive/MyDrive/_AgenticIslam/_Parsed Books Production/Manuscript/Manuscript_pages.json


In [ ]:
# @title Calcualate Total usage of reading the book
import json

def calculate_gemini_cost(usage_data: dict, model_name: str) -> float:
    """
    Calculates the total cost based on the number of input and output tokens
    for a specified Gemini model.
    """
    # Pricing per 1 million tokens in USD
    pricing = {
        'flash-2.5': {
            'input': 0.30,  # $0.30 per 1M input tokens
            'output': 2.50, # $2.50 per 1M output tokens
        },
        'flash-lite-2.5': {
            'input': 0.10,  # $0.10 per 1M input tokens
            'output': 0.40, # $0.40 per 1M output tokens
        },
        # Gemini 2.5 Pro pricing is tiered based on input token count
        'pro-2.5_tier1': { # <= 200k tokens
            'input': 1.25,
            'output': 10.00
        },
        'pro-2.5_tier2': { # > 200k tokens
            'input': 2.50,
            'output': 15.00
        }
    }

    # Get the token counts from the provided usage data
    try:
        input_tokens = usage_data['input_tokens']
        output_tokens = usage_data['output_tokens']
    except KeyError as e:
        print(f"Error: Missing key in usage_data object - {e}")
        return 0.0

    # Determine the pricing tier for Gemini 2.5 Pro based on input tokens
    if model_name == 'pro-2.5':
        if input_tokens > 200000:
            model_pricing = pricing['pro-2.5_tier2']
        else:
            model_pricing = pricing['pro-2.5_tier1']
    elif model_name in ['flash-2.5', 'flash-lite-2.5']:
        model_pricing = pricing[model_name]
    else:
        raise ValueError(f"Unknown model: '{model_name}'. Please use 'flash-2.5', 'flash-lite-2.5', or 'pro-2.5'.")

    # Calculate the cost
    cost = (input_tokens / 1_000_000) * model_pricing['input'] + \
           (output_tokens / 1_000_000) * model_pricing['output']

    return round(cost, 4)

# Example usage
if __name__ == '__main__':
    total_usage_object = output["Total_usageMetadata"]
    parsed_usage_data = {
        "input_tokens": total_usage_object["promptTokenCount"],
        "output_tokens": total_usage_object["candidatesTokenCount"]
    }

    print(f"Usage Data (Parsed): {json.dumps(parsed_usage_data, indent=2)}\n")

    try:
        # Calculate cost for Flash 2.5
        flash_cost = calculate_gemini_cost(parsed_usage_data, 'flash-2.5')
        output["Total_usageMetadata"]["parsing_cost"] =flash_cost
        output["Total_usageMetadata"]["model"] ="flash-2.5"
        print(f"Calculated cost for 'flash-2.5': ${flash_cost}")

        # Calculate cost for Flash 2.5 Lite
        flash_lite_cost = calculate_gemini_cost(parsed_usage_data, 'flash-lite-2.5')
        print(f"Calculated cost for 'flash-lite-2.5': ${flash_lite_cost}")

        # Calculate cost for Pro 2.5
        pro_cost = calculate_gemini_cost(parsed_usage_data, 'pro-2.5')
        print(f"Calculated cost for 'pro-2.5': ${pro_cost}")

    except ValueError as e:
        print(e)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

print("------")
print(output["Total_usageMetadata"])

Usage Data (Parsed): {
  "input_tokens": 757670,
  "output_tokens": 491235
}

Calculated cost for 'flash-2.5': $1.4554
Calculated cost for 'flash-lite-2.5': $0.2723
Calculated cost for 'pro-2.5': $9.2627
------
{'totalTokenCount': 1248905, 'candidatesTokenCount': 491235, 'promptTokenCount': 757670, 'parsing_cost': 1.4554, 'model': 'flash-2.5'}


In [ ]:
# @title Fill infromation of Books in Firestore
parent_ref=db.collection("books").document(BOOK_ID)
doc_ref = parent_ref.collection("parts").document(f"{BOOK_PART_NUMBER}")

parent_data_book = {
    "book_id": BOOK_ID,
    "book_author": BOOK_AUTHOR,
    "books_url": {f"{BOOK_PART_NUMBER}": BOOK_URL_FIRESTORE},
    "toc_offset":TOC_OFFEST,
    'search_range': SEARCH_RANGE,
}
parent_ref.set(parent_data_book, merge=True)

data_book = {
    "usage_parsing": output["Total_usageMetadata"],
    "book_id": BOOK_ID,
    "book_part": BOOK_PART_NUMBER,
    "book_title": BOOK_TITLE_PRODUCTION,
    "book_subtitle": BOOK_SUBTITLE_PRODUCTION,
}
print(data_book)

doc_ref.set(data_book, merge=True)


{'usage_parsing': {'totalTokenCount': 1248905, 'candidatesTokenCount': 491235, 'promptTokenCount': 757670, 'parsing_cost': 1.4554, 'model': 'flash-2.5'}, 'book_id': 'fiqh_001-123456', 'book_part': 8, 'book_title': 'موسوعة الفقه الإسلامي والقضايا المعاصرة', 'book_subtitle': ''}


update_time {
  seconds: 1760960125
  nanos: 190662000
}

# Prompts for TOC

In [ ]:
# @title Prompt to Extract the TOC
# Use Google Gemnin 2.5 pro with the following prompt:
Prompt = """
Parse the entire PDF document and convert its contents into **Markdown format**.

## Key Requirements
* **Extract all items and their corresponding page numbers.**
* **Maintain the original hierarchical structure** of the document.
* Avoid adding any citation to the extracted markdown. like this: [cite_start] ... [cite: 63]
* Make sure that the page number extracted in the Arabic numerals 1,2...9
* Strictly follow the structure of example output below


## Example output:
* **مقدمة الطبعة الجديدة**: 17
* **تقديم**: 21
* **منهج هذا الكتاب**: 23
* **الباعث المباشر على تأليف هذا الكتاب**: 28
* **مقدمات ضرورية عن الفقه**: 30
    * **المطلب الأول: معنى الفقه وخصائصه**: 30
        * **خصائص الفقه**: 32
    * **المطلب الثاني: لمحة موجزة عن فقهاء المذاهب**: 41
    * **المطلب الثالث: مراتب الفقهاء وكتب الفقه**: 58
    * **المطلب الرابع: اصطلاحات الفقه والمؤلفين فيه**: 61
        * **أولاً: المصطلحات الفقهية العامة**: 62
        * **ثانياً: المصطلحات الخاصة بالمذاهب**: 66
            * **مصطلحات المذهب الحنفي**: 67
            * **مصطلحات المذهب المالكي**: 69
            * **مصطلحات المذهب الشافعي**: 70
            * **مصطلحات المذهب الحنبلي**: 74
    * **المطلب الخامس: أسباب اختلاف الفقهاء**: 76
    * **المطلب السادس: الضوابط الشرعية للأخذ بأيسر المذاهب**: 80
        * **تمهيد**: 80
        * **خطة البحث**: 82
        * **الفرع الأول: المذاهب أو الآراء التي يمكن الأخذ بها**: 82
        * **الفرع الثاني: هل التزام مذهب معين أمر مطلوب أصولياً؟**: 84
        * **الفرع الثالث: هل يجب سؤال الأفضل والأرجح في العلم**: 87
        * **الفرع الرابع: آراء الأصوليين في مسألة اختيار الأيسر**: 89
        * **الفرع الخامس: أنواع الضوابط الشرعية للأخذ بأيسر المذاهب**: 105
    * **المطلب السابع: المصيب في الاجتهاد**: 118
    * **المطلب الثامن: طريقة الاجتهاد**: 120
    * **المطلب التاسع: نقض الاجتهاد وتغييره وتغير الأحكام بتغير الأزمان**: 120
    * **المطلب العاشر: خطة البحث**: 123
    * **المطلب الحادي عشر: جدول المقاييس**: 123
"""

In [ ]:
# @title Example of the Deisred flat JSON

json_output ="""
[
  {
    "id": "item_1",
    "level": 1,
    "title": "مقدمة الطبعة الجديدة",
    "startPage": 17,
    "endPage": 20,
    "parentId": null,
    "pathIds": [
      "item_1"
    ],
    "pathTitles": [
      "مقدمة الطبعة الجديدة"
    ],
    "is_confident": true
  },
  {
    "id": "item_2",
    "level": 1,
    "title": "تقديم",
    "startPage": 21,
    "endPage": 22,
    "parentId": null,
    "pathIds": [
      "item_2"
    ],
    "pathTitles": [
      "تقديم"
    ],
    "is_confident": true
  },
  {
    "id": "item_3",
    "level": 1,
    "title": "منهج هذا الكتاب",
    "startPage": 23,
    "endPage": 27,
    "parentId": null,
    "pathIds": [
      "item_3"
    ],
    "pathTitles": [
      "منهج هذا الكتاب"
    ],
    "is_confident": true
  },
  {
    "id": "item_4",
    "level": 1,
    "title": "الباعث المباشر على تأليف هذا الكتاب",
    "startPage": 28,
    "endPage": 29,
    "parentId": null,
    "pathIds": [
      "item_4"
    ],
    "pathTitles": [
      "الباعث المباشر على تأليف هذا الكتاب"
    ],
    "is_confident": true
  },
"""

In [ ]:
import json
import os

# The data containing 'pages_list' is in the 'output' variable
# from the previous execution of cell 4dONPXHXdnyn.
# We don't need to reload the batch result JSON here to get pages.

with open("/content/Manuscript_1-job_batch_result.json", 'r', encoding='utf-8') as f:
            output = json.load(f)
            print(output)

for o in output[:5]:
  print(o)

[{'page_content': "# وقفية الأمير غازي للفكر القراني\n\nTHE PRINCE GHAZI FOR QUR'ANIC THOUGHT\n\n## هذا شرح شيخ مشايخنا الفاضل\n\nالعلم المشهور المحقق الشيخ\nمنصور المنوفي لمنظومته\nفي الموجهات\nعلى الله عنه\nامين", 'notes': {'footnote': 'This file was downloaded from QuranicThought.com', 'marginalia': 'دخلت في ملك الفقير\nمحمدها هر الحسيني\nعفر له'}, 'page_number': '1'}, {'page_content': '# بسم الله الرحمن الرحيم\n\nإن خير ما توجهت به الآمال إلى تحصيل المطالب وأولى ما توجهت به صدور الأعمال فبلغت أعلام المراتب، حمد الله الذي أبدع الكائنات على غير سابق منوال. وغير الموجودات من بحار كرمه بواسع العطاء والنوال ودوايم ديم الصلاة والسلام والتسليم على المختص دون ساير الممكنات بخواص التكريم محمد الذي انتشرت قضايا عدله في جميع الأنام وانتشرت فضله على العالمين في جميع الأوقات والأيام. وعلى آله الذين آل إليهم من غاية الشرف وحازوا باتباعهم في الجنة أعلا الغرف. وبعد فقد كنت طالعت مع جماعة من الإخوان شرح التهذيب على حسب ما يقتضيه التعليم والتدريب فعندما أعاننا الله على إتمام وأقدرنا على الظفر بفض خت

In [ ]:
import json
import os

def create_simple_markdown_book(json_file_path: str, output_md_path: str):
    """
    Reads a JSON file containing book pages, then formats and writes
    the content to a single Markdown (.md) file with page content and number.

    Expected JSON structure: a list of dictionaries, where each dictionary
    represents a page and has at least 'page_content' and 'page_number' keys.

    Args:
        json_file_path (str): The full path to the input JSON file.
        output_md_path (str): The full path where the output Markdown file will be saved.
    """
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: JSON file not found at {json_file_path}")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {json_file_path}")
        return
    except Exception as e:
        print(f"An unexpected error occurred while reading the JSON: {e}")
        return

    if not isinstance(data, list):
        print(f"Error: Expected JSON to be a list of pages, but got {type(data)}")
        return

    md_content = []

    for page in data:
        # Extract fields, defaulting to empty string for safety
        page_content = page.get('page_content', '').strip()
        page_number = page.get('page_number', '') # Keep page number as is, don't strip if it's just digits

        # Compile the content for the current page:
        # 1. page content
        # 2. page number on a new line
        # No extra dividers as per the simplified request

        page_block = f"{page_content}\n{page_number}\n"

        # Add the page block to the list
        md_content.append(page_block)

    # 4. Write the compiled content to the Markdown file
    try:
        final_md_text = "\n".join(md_content)

        with open(output_md_path, 'w', encoding='utf-8') as f:
            f.write(final_md_text)

        print(f"Successfully created Markdown book file at: {output_md_path}")

    except Exception as e:
        print(f"An error occurred while writing the Markdown file: {e}")

You can use the `create_simple_markdown_book` function like this, specifying the path to your JSON file and the desired output path for the Markdown file:

In [ ]:
import json
from typing import List, Dict, Any

def create_markdown_book(pages: List[Dict[str, Any]]) -> str:
    """
    Loops through a list of page data (in JSON/Dict format) and
    creates a single Markdown-formatted string representing a book.

    Args:
        pages: A list of dictionaries, where each dictionary contains
               'page_content', 'notes' (with 'footnote' and 'marginalia'),
               and 'page_number'.

    Returns:
        A single string containing the entire book content in Markdown format.
    """
    markdown_output = []

    for page_data in pages:
        # 1. Start with the main content of the page
        content = page_data.get('page_content', 'No Content Found')
        markdown_output.append(content)
        markdown_output.append('\n') # Add a line break after the content

        # 2. Add a horizontal rule to separate pages clearly in the Markdown
        markdown_output.append('---')
        markdown_output.append('\n')

        # 3. Add page notes/metadata
        notes = page_data.get('notes', {})
        page_num = page_data.get('page_number', 'N/A')

        # Add a section for page number and notes
        markdown_output.append(f'## Page {page_num} Metadata')
        markdown_output.append('\n')

        if notes:
            # Add Footnote
            footnote = notes.get('footnote')
            if footnote:
                # Use blockquote style for notes/footnotes to set them apart
                markdown_output.append(f'> **Footnote:** {footnote}')
                markdown_output.append('\n')

            # Add Marginalia
            marginalia = notes.get('marginalia')
            if marginalia:
                markdown_output.append(f'> **Marginalia:** {marginalia.replace("\n", " ")}') # Replace internal newlines for better display
                markdown_output.append('\n')
        else:
            markdown_output.append('> *No additional notes for this page.*')
            markdown_output.append('\n')

        # Add a final separation between the metadata block and the next page's main content
        markdown_output.append('***')
        markdown_output.append('\n\n')

    # Join all parts into a single string
    return ''.join(markdown_output)

temp_pages_json_path = "/content/Manuscript_1-job_batch_result.json"
if 'pages_list' in output and isinstance(output['pages_list'], list):
    with open(temp_pages_json_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

# Generate the Markdown content
markdown_book_content = create_markdown_book(output)

# Print the result (or save it to a file)
print(markdown_book_content)

# To save to a file:
with open('markdown_book.md', 'w', encoding='utf-8') as f:
    f.write(markdown_book_content)

# وقفية الأمير غازي للفكر القراني

THE PRINCE GHAZI FOR QUR'ANIC THOUGHT

## هذا شرح شيخ مشايخنا الفاضل

العلم المشهور المحقق الشيخ
منصور المنوفي لمنظومته
في الموجهات
على الله عنه
امين
---
## Page 1 Metadata
> **Footnote:** This file was downloaded from QuranicThought.com
> **Marginalia:** دخلت في ملك الفقير محمدها هر الحسيني عفر له
***

# بسم الله الرحمن الرحيم

إن خير ما توجهت به الآمال إلى تحصيل المطالب وأولى ما توجهت به صدور الأعمال فبلغت أعلام المراتب، حمد الله الذي أبدع الكائنات على غير سابق منوال. وغير الموجودات من بحار كرمه بواسع العطاء والنوال ودوايم ديم الصلاة والسلام والتسليم على المختص دون ساير الممكنات بخواص التكريم محمد الذي انتشرت قضايا عدله في جميع الأنام وانتشرت فضله على العالمين في جميع الأوقات والأيام. وعلى آله الذين آل إليهم من غاية الشرف وحازوا باتباعهم في الجنة أعلا الغرف. وبعد فقد كنت طالعت مع جماعة من الإخوان شرح التهذيب على حسب ما يقتضيه التعليم والتدريب فعندما أعاننا الله على إتمام وأقدرنا على الظفر بفض ختامه سألني من لا تسعني مخالفته وتجب على مساعدته ومساعدته